In [ ]:
import numpy
import torch
import pandas as pd
from scipy import sparse

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

df = pd.read_csv('data_goodreads.csv')

In [ ]:
# TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_tfidf_sparse = tfidf_vectorizer.fit_transform(df["summary"])

sparse.save_npz('./data/tfidf_emb.npz', X_tfidf_sparse)

In [ ]:
# bag-of-words

bow_vectorizer = CountVectorizer(max_features=5000,stop_words="english")
X_bow_sparse = bow_vectorizer.fit_transform(df["summary"])

sparse.save_npz('./data/bow_emb.npz', X_bow_sparse)

In [ ]:
import gensim.downloader

w2v_google = gensim.downloader.load('word2vec-google-news-300')

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embeddings_list = torch.FloatTensor(w2v_google.vectors).to(device)
embeddings_list = torch.nn.Embedding.from_pretrained(weights)

def get_sentence_vector_gpu(sentence, model, embedding_layer):
    if not isinstance(sentence, str):
        return torch.zeros(model.vector_size)

    words_ids = [model.key_to_index[w] for w in sentence.split() if w in model.key_to_index]

    if not words_ids:
        return torch.zeros(model.vector_size)

    with torch.no_grad():
        words_ids_tensor = torch.LongTensor(indices).to(device)
        word_vectors = embedding_layer(words_ids_tensor)
        sentence_vector = torch.mean(word_vectors, dim=0).cpu() # błąd bez .cpu(), numpy sobie nie radzi?

    return sentence_vector

df["w2v"] = df["summary"].apply(lambda x: get_sentence_vector_gpu(str(x), w2v_google, embedding).numpy())
all_w2v_vectors = np.array(df['w2v'].tolist())
np.savez_compressed(npz_file_path, w2v_vectors='./data/w2v_vectors.npz')